In [1]:
# Section 1A — The raw documents (fictional company: NovaMind Technologies)

documents = [
    """NovaMind Technologies was founded in 2019 by Dr. Priya Chakraborty and Marcus Okonkwo
    in Waterloo, Ontario. The company specializes in AI-powered diagnostic tools for
    veterinary medicine. Their flagship product, PetScan Pro, uses computer vision to
    analyze X-rays and MRI scans of animals, detecting fractures, tumors, and joint
    abnormalities with 94.2% accuracy. PetScan Pro is currently used in over 350
    veterinary clinics across Canada and the United States.""",

    """NovaMind raised a $12 million Series A round in March 2021, led by Horizon Ventures
    with participation from BioTech Capital and the Ontario Innovation Fund. A subsequent
    $38 million Series B closed in January 2023, led by Greenfield Partners. The company
    has been cash-flow positive since Q3 2023. Total headcount as of 2024 is 127 employees,
    with 74 based in Waterloo and 53 working remotely across North America.""",

    """PetScan Pro works by first preprocessing the uploaded medical image using a custom
    denoising algorithm called ClearVet. The cleaned image is then passed through a
    fine-tuned ResNet-152 backbone for feature extraction. Classification is handled by a
    proprietary ensemble of three models: a vision transformer (ViT-L), a convolutional
    network, and a lightweight decision tree that acts as a tiebreaker. Results are returned
    in under 8 seconds with a detailed heatmap overlay showing areas of concern.""",

    """In September 2024, NovaMind announced FeatherCheck, a new product designed for avian
    diagnostics. FeatherCheck analyzes feather samples and blood work photographs to screen
    for common diseases in parrots, eagles, and poultry. Early trials at the University of
    Guelph showed 87.5% accuracy in detecting Psittacine Beak and Feather Disease (PBFD)
    from feather images alone. FeatherCheck is expected to launch commercially in Q2 2025.""",

    """Dr. Priya Chakraborty holds a PhD in Biomedical Engineering from the University of
    Toronto and previously worked at Google Health for three years. Marcus Okonkwo has a
    Masters in Computer Science from the University of Waterloo and was a founding engineer
    at Shopify's machine learning division. The two met at a veterinary AI hackathon in 2018
    and decided to start NovaMind after winning first place with a prototype that could
    detect hip dysplasia in dogs from smartphone photos.""",

    """NovaMind's engineering team follows a rigorous model evaluation protocol. Every model
    update must pass a 4-stage pipeline: unit tests on synthetic data, validation on a held-out
    dataset of 15,000 annotated images, a red-team adversarial review by the safety team, and
    finally a 2-week shadow deployment where the new model runs alongside the production model
    and outputs are compared. Only if the new model matches or exceeds production accuracy with
    no increase in false negatives is it promoted to production.""",

    """The company's data infrastructure runs on a hybrid cloud setup. Training workloads use
    a cluster of 8 NVIDIA A100 GPUs hosted on AWS, while inference for PetScan Pro runs on
    edge devices — specifically NVIDIA Jetson Orin modules deployed directly in partner clinics.
    This edge deployment ensures that sensitive animal medical data never leaves the clinic's
    network, addressing privacy concerns raised by several provincial veterinary boards.""",

    """NovaMind's main competitors include VetAI (based in San Francisco, raised $45M, focused
    on canine diagnostics only), AnimalSense (London-based, uses ultrasound analysis), and
    PawPath Diagnostics (a subsidiary of a large pharmaceutical company). NovaMind differentiates
    itself through multi-species support, edge deployment for data privacy, and their ensemble
    architecture which consistently outperforms single-model approaches in independent benchmarks
    published by the Veterinary AI Consortium."""
]

print(f"Knowledge base: {len(documents)} documents")
print(f"Total characters: {sum(len(d) for d in documents):,}")


Knowledge base: 8 documents
Total characters: 3,918


# Section 1 — Chunking

In [2]:

def chunk_text(text, chunk_size=200, overlap=50):
    """
    Simple character-based chunking with overlap.

    Inputs:
      - text: str
      - chunk_size: target number of characters per chunk
      - overlap: number of characters to overlap between consecutive chunks

    Returns:
      - list[str]: chunks of text
    """
    assert overlap < chunk_size
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        # move start forward with overlap
        start = end - overlap

    return chunks


In [3]:
# Section 1B — chunk_text function
def chunk_text(text, chunk_size=200, overlap=50):
    assert overlap < chunk_size
    chunks = []
    start = 0
    n = len(text)

    while start < n:
        end = min(start + chunk_size, n)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == n:
            break
        start = end - overlap

    return chunks


# Section 1B — build all_chunks from all documents
all_chunks = []
chunk_sources = []

for doc_idx, doc in enumerate(documents):
    chunks = chunk_text(doc, chunk_size=200, overlap=50)
    all_chunks.extend(chunks)
    chunk_sources.extend([doc_idx] * len(chunks))

print(f"Total chunks: {len(all_chunks)}")
print()
print("Example chunk:")
print(all_chunks[0])


Total chunks: 28

Example chunk:
NovaMind Technologies was founded in 2019 by Dr. Priya Chakraborty and Marcus Okonkwo 
    in Waterloo, Ontario. The company specializes in AI-powered diagnostic tools for 
    veterinary medicine. Th


# Section 2 — Embeddings + Vector Store
### 2A — Load

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the sentence-transformers embedding model (LLM1)
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Quick sanity check (should print a vector shape like (384,))
test_embedding = embedding_model.encode("This is a test sentence.")
print("Embedding shape:", np.array(test_embedding).shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (384,)


## Section 2B — Embed all chunks and build the vector store

In [5]:
chunk_embeddings = None

if embedding_model is None:
    print("Load embedding_model first.")
elif len(all_chunks) == 0:
    print("Create chunks first (all_chunks is empty).")
else:
    chunk_embeddings = embedding_model.encode(all_chunks)
    chunk_embeddings = np.array(chunk_embeddings)
    print(f"Vector store shape: {chunk_embeddings.shape}")
    print(f"That's {chunk_embeddings.shape[0]} chunks, each represented by {chunk_embeddings.shape[1]} numbers.")


Vector store shape: (28, 384)
That's 28 chunks, each represented by 384 numbers.


## 2C — Cosine similarity helper + quick test


In [6]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors using numpy.

    Returns a float in [-1, 1]. If either vector has zero magnitude, returns 0.0.
    """
    a = np.array(a)
    b = np.array(b)
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


# TODO: Lets test it out! (This is not the final application) Create your own three test sentences
sentences = [
    "NovaMind develops AI tools for veterinary diagnostics.",
    "PetScan Pro is a product for analyzing animal medical images.",
    "I like to play basketball on the weekends."
]

if embedding_model is None:
    print("Load embedding_model first.")
else:
    vecs = embedding_model.encode(sentences)
    vecs = np.array(vecs)

    sim01 = cosine_similarity(vecs[0], vecs[1])
    sim02 = cosine_similarity(vecs[0], vecs[2])
    sim12 = cosine_similarity(vecs[1], vecs[2])

    print("Similarity(0,1):", sim01)
    print("Similarity(0,2):", sim02)
    print("Similarity(1,2):", sim12)

    print("\nExpected: (0,1) should be the highest similarity.")


Similarity(0,1): 0.5181401968002319
Similarity(0,2): 0.016330605372786522
Similarity(1,2): -0.0314156748354435

Expected: (0,1) should be the highest similarity.


## Section 3 — Retrieval

In [7]:
def retrieve(query, chunks, chunk_embeddings, embedding_model, top_k=3):
    """Return the top_k most relevant chunks for `query`.

    Returns list of tuples: (chunk_text, similarity_score, chunk_index).
    """
    if embedding_model is None:
        raise ValueError("embedding_model is None.")
    if chunk_embeddings is None or len(chunk_embeddings) == 0:
        raise ValueError("chunk_embeddings is empty.")
    if len(chunks) != chunk_embeddings.shape[0]:
        raise ValueError("len(chunks) and chunk_embeddings.shape[0] must match.")

    # 1) Embed the query
    query_vec = embedding_model.encode([query])
    query_vec = np.array(query_vec)[0]  # shape (D,)

    # 2) Compute cosine similarity vs every chunk
    sims = []
    for i in range(chunk_embeddings.shape[0]):
        sim = cosine_similarity(query_vec, chunk_embeddings[i])
        sims.append((sim, i))

    # 3) Sort by similarity (highest first) and take top_k
    sims.sort(key=lambda x: x[0], reverse=True)
    top = sims[:top_k]

    # 4) Build output tuples
    results = []
    for sim, idx in top:
        results.append((chunks[idx], sim, idx))

    return results

# Safety check
if chunk_embeddings is None:
    print("Build chunk_embeddings first.")
else:
    print("Ready to retrieve. (Implement `retrieve` above.)")


Ready to retrieve. (Implement `retrieve` above.)


# Section 4 — Generation

### 4A — Load tokenizer + generation model (flan‑t5)

In [8]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load tokenizer + generation model for google/flan-t5-base
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
gen_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

if tokenizer is None or gen_model is None:
    print("tokenizer and/or gen_model is None — load them using HF docs.")
else:
    print("Generation model loaded.")


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generation model loaded.


### 4B — Build the prompt

In [9]:
def build_prompt(query, retrieved_chunks):
    """Assemble the final prompt from the question and retrieved context.

    retrieved_chunks: list of (chunk_text, similarity_score, chunk_index)
    """
    # Build a nicely formatted context block with IDs
    context_parts = []
    for i, (chunk_text, score, idx) in enumerate(retrieved_chunks):
        context_parts.append(
            f"[chunk_{idx}] (similarity: {score:.4f})\n{chunk_text}"
        )
    context_block = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a helpful assistant answering questions about NovaMind Technologies.

Use ONLY the information provided in the context below.
If the answer is not in the context, say "I don't know based on the provided documents."

Question:
{query}

Context:
{context_block}

Answer the question in 1–3 sentences and, where possible, reference the relevant chunk IDs in square brackets.
"""
    return prompt

# Quick print to see what the prompt looks like
demo_query = "What does NovaMind sell?"
if embedding_model is None or chunk_embeddings is None or len(all_chunks) == 0:
    print("Build your index first.")
else:
    demo_retrieved = retrieve(demo_query, all_chunks, chunk_embeddings, embedding_model, top_k=3)
    prompt = build_prompt(demo_query, demo_retrieved)
    print("=== THE FULL PROMPT SENT TO THE MODEL ===")
    print(prompt[:1200], "..." if len(prompt) > 1200 else "")
    print("=========================================")


=== THE FULL PROMPT SENT TO THE MODEL ===
You are a helpful assistant answering questions about NovaMind Technologies.

Use ONLY the information provided in the context below. 
If the answer is not in the context, say "I don't know based on the provided documents."

Question:
What does NovaMind sell?

Context:
[chunk_3] (similarity: 0.6854)
NovaMind raised a $12 million Series A round in March 2021, led by Horizon Ventures 
    with participation from BioTech Capital and the Ontario Innovation Fund. A subsequent 
    $38 million Series B

---

[chunk_24] (similarity: 0.6714)
NovaMind's main competitors include VetAI (based in San Francisco, raised $45M, focused 
    on canine diagnostics only), AnimalSense (London-based, uses ultrasound analysis), and 
    PawPath Diagnos

---

[chunk_0] (similarity: 0.6198)
NovaMind Technologies was founded in 2019 by Dr. Priya Chakraborty and Marcus Okonkwo 
    in Waterloo, Ontario. The company specializes in AI-powered diagnostic tools for 
    vet

### 4C — generate_answer implementation

In [10]:
def generate_answer(prompt, tokenizer, model, max_new_tokens=200, temperature=0.7, do_sample=True):
    """Generate an answer using a local Hugging Face model (seq2seq)."""
    # 1) tokenize
    inputs = tokenizer(prompt, return_tensors="pt")

    # 2) generate
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
    }
    if do_sample:
        gen_kwargs["temperature"] = temperature
    else:
        # greedy: do_sample=False, no temperature needed
        gen_kwargs["temperature"] = None

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask", None),
        **{k: v for k, v in gen_kwargs.items() if v is not None},
    )

    # 3) decode
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

print("generate_answer stub ready (implement it using HF docs).")


generate_answer stub ready (implement it using HF docs).


### Section 5 — Full rag_query Pipeline

In [11]:
def rag_query(question, chunks, chunk_embeddings, embedding_model, tokenizer, gen_model,
              top_k=3, temperature=0.7, show_context=False):
    """Full RAG pipeline: question -> retrieved context -> prompt -> generated answer."""
    if embedding_model is None:
        raise ValueError("embedding_model is None.")
    if tokenizer is None or gen_model is None:
        raise ValueError("tokenizer/gen_model not loaded.")
    if chunk_embeddings is None or len(chunks) == 0:
        raise ValueError("No chunks / embeddings available.")

    # 1) Retrieve
    retrieved = retrieve(question, chunks, chunk_embeddings, embedding_model, top_k=top_k)

    # 2) Build prompt
    prompt = build_prompt(question, retrieved)

    # 3) Generate answer
    answer = generate_answer(
        prompt,
        tokenizer,
        gen_model,
        max_new_tokens=200,
        temperature=temperature,
        do_sample=True,      # you can toggle for experiments
    )

    if show_context:
        print("=== Retrieved context ===")
        for i, (chunk, score, idx) in enumerate(retrieved):
            print(f"[{i+1}] chunk_{idx} (similarity: {score:.4f})")
            print_wrapped(chunk)
            print()
        print("=========================\n")

    return answer

print("rag_query stub ready (implement it after retrieve/build_prompt/generate_answer).")


rag_query stub ready (implement it after retrieve/build_prompt/generate_answer).


## Section 6 — Experiments (minimal scaffolding)

### Experiment 1 — Chunking paradox


In [12]:
# Section 6 — Experiment 1 — Chunking Paradox

questions_e1 = [
    "Who founded NovaMind and when?",
    "What does FeatherCheck do?",
    "How does NovaMind handle data privacy?"
]

for size in [50, 200, 500]:
    overlap = min(20, size // 4)   # overlap is always smaller than chunk_size
    print(f"\n===== chunk_size={size}, overlap={overlap} =====")

    all_chunks_e1 = []
    chunk_sources_e1 = []

    for doc_idx, doc in enumerate(documents):
        chunks_tmp = chunk_text(doc, chunk_size=size, overlap=overlap)
        all_chunks_e1.extend(chunks_tmp)
        chunk_sources_e1.extend([doc_idx] * len(chunks_tmp))

    chunk_embeddings_e1 = np.array(embedding_model.encode(all_chunks_e1))

    for q in questions_e1:
        print(f"\nQ: {q}")
        ans = rag_query(
            q,
            all_chunks_e1,
            chunk_embeddings_e1,
            embedding_model,
            tokenizer,
            gen_model,
            top_k=3,
            temperature=0.7,
            show_context=False,
        )
        print("A:", ans)



===== chunk_size=50, overlap=12 =====

Q: Who founded NovaMind and when?
A: It will likely involve three years from

Q: What does FeatherCheck do?
A: announced In xavier 20024 Nov 2015 It should still go under (it needs testing out for its own uses: people may want for life. There seem too high technical and market forces (in aviation at time with very poor runway forecast as that becomes necessary depending in the age gap which exists during pilot age group is determined sole time, according ) The latest data should enable Nova mindatas and data sources

Q: How does NovaMind handle data privacy?
A: species backup technology (with inherently unsubtracised hardware to safeguard logging),edgedeploying with deep understanding and useless training modules to develop individual-use product ideas using these technology concepts at an edge setting to achieve greater enterprise through the implementation

===== chunk_size=200, overlap=20 =====

Q: Who founded NovaMind and when?
A: M Y C Chanh

In [14]:
# Experiment 2 — your code here
import textwrap

def print_wrapped(text, width=90):
    print(textwrap.fill(text, width=width))

question_e2 = "How does PetScan Pro process images?"

for k in [1, 3, 10]:
    print(f"\n===== top_k={k} =====")
    ans = rag_query(
        question_e2,
        all_chunks,
        chunk_embeddings,
        embedding_model,
        tokenizer,
        gen_model,
        top_k=k,
        temperature=0.7,
        show_context=True,  # so you can see noise vs coverage
    )
    print("Answer:", ans)



===== top_k=1 =====
=== Retrieved context ===
[1] chunk_6 (similarity: 0.7019)
PetScan Pro works by first preprocessing the uploaded medical image using a custom
denoising algorithm called ClearVet. The cleaned image is then passed through a      fine-
tuned ResNet-152 backb


Answer: denover in advance The done layer undergo special compression, cleaning operations including fillover after each session of this algorithm using several back and fill fill phases from your image storage unit across digital layers This also createa very similar

===== top_k=3 =====


Token indices sequence length is longer than the specified maximum sequence length for this model (666 > 512). Running this sequence through the model will result in indexing errors


=== Retrieved context ===
[1] chunk_6 (similarity: 0.7019)
PetScan Pro works by first preprocessing the uploaded medical image using a custom
denoising algorithm called ClearVet. The cleaned image is then passed through a      fine-
tuned ResNet-152 backb

[2] chunk_1 (similarity: 0.6686)
diagnostic tools for      veterinary medicine. Their flagship product, PetScan Pro, uses
computer vision to      analyze X-rays and MRI scans of animals, detecting fractures,
tumors, and joint      a

[3] chunk_22 (similarity: 0.5478)
ence for PetScan Pro runs on      edge devices — specifically NVIDIA Jetson Orin modules
deployed directly in partner clinics.      This edge deployment ensures that sensitive
animal medical data neve


Answer: Custom DENTAlling EXCEPT thin the files with DTN(CT0809010): its pre processing software use the NAVI ENTANT software system is deployed securely near clinic access paths including doctors. When approved there now would actually be additional pre processor program

In [15]:
# Experiment 3 — your code here

def build_prompt_zeroshot(query, retrieved_chunks):
    return build_prompt(query, retrieved_chunks)

def build_prompt_fewshot(query, retrieved_chunks):
    """Add 1–2 example Q/A pairs before the real question."""
    examples = """Example 1:
Question: Who founded NovaMind?
Context: [chunk_0] NovaMind Technologies was founded in 2019 by Dr. Priya Chakraborty and Marcus Okonkwo in Waterloo, Ontario.
Answer: NovaMind was founded in 2019 by Dr. Priya Chakraborty and Marcus Okonkwo. [chunk_0]

Example 2:
Question: What is FeatherCheck?
Context: [chunk_x] FeatherCheck is NovaMind's product for avian diagnostics, analyzing feather samples and blood work photos.
Answer: FeatherCheck is NovaMind's AI product for avian diagnostics, analyzing feather and blood images. [chunk_x]
"""

    context_parts = []
    for (chunk_text, score, idx) in retrieved_chunks:
        context_parts.append(f"[chunk_{idx}] (similarity: {score:.4f})\n{chunk_text}")
    context_block = "\n\n---\n\n".join(context_parts)

    prompt = f"""{examples}

Now answer a new question using the same style.

Question:
{query}

Context:
{context_block}

Answer using only the context and cite chunk IDs in square brackets.
"""
    return prompt

question_e3 = "How does NovaMind ensure its models are safe before deployment?"

retrieved_e3 = retrieve(question_e3, all_chunks, chunk_embeddings, embedding_model, top_k=3)

# Zero-shot
prompt_zero = build_prompt_zeroshot(question_e3, retrieved_e3)
ans_zero = generate_answer(prompt_zero, tokenizer, gen_model, max_new_tokens=200, temperature=0.7, do_sample=True)
print("=== Zero-shot ===")
print(ans_zero)
print()

# Few-shot
prompt_few = build_prompt_fewshot(question_e3, retrieved_e3)
ans_few = generate_answer(prompt_few, tokenizer, gen_model, max_new_tokens=200, temperature=0.7, do_sample=True)
print("=== Few-shot ===")
print(ans_few)


=== Zero-shot ===
evaluation

=== Few-shot ===
ensure your information (the platform used are 100% human available and reliable while no third source, virus program), data manipulation
